# 04 (Block 3) — MCP Tool Interoperability & Pausable Human-in-the-Loop Gates
**Official Reference Pattern**: Google 5-Day AI Agents Intensive (`day-2b-agent-tools-best-practices.ipynb`)
**Antigravity IDE Usage**: Read-only reference for **Block 3 (MCP SDK & Verifiable Workflows)**.

## What to Observe
1. **`McpToolset` & Stdio/SSE Connection**: Wrapping external Model Context Protocol servers with an explicit `tool_filter` allowlist (*Least Privilege*).
2. **Human-in-the-Loop Confirmation Gate (`ToolContext.request_confirmation`)**: Implementing **Slow Engineering Principle 01 (*AI builds, it never signs*)** and **Principle 10 (*Every Input is a Stranger*)** in code.


In [2]:
from google.adk.agents import Agent
from google.adk.tools.mcp_tool import McpToolset, StdioConnectionParams
from google.adk.tools import ToolContext
from mcp import StdioServerParameters

# 1. Connect to local MCP Server with an explicit tool_filter allowlist
mcp_tools = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="npx",
            args=["tsx", "mcp-server.ts"],  # your MCP server (built in Lab 2)
        ),
        timeout=15,
    ),
    tool_filter=["search_architecture_docs", "draft_engineering_ticket"],  # Least-privilege allowlist
)

# 2. High-Consequence Tool with Mandatory Human Confirmation Gate (Principle 01 & 03)
def deploy_migration_to_production(migration_id: str, dry_run_passed: bool, tool_context: ToolContext) -> dict:
    """Requires explicit human approval before executing any production migration."""
    if not dry_run_passed:
        return {"status": "BLOCKED", "reason": "Fail Soft: dry_run_passed must be True."}

    if not tool_context.tool_confirmation:
        tool_context.request_confirmation(
            hint=f"Approve production migration {migration_id}?",
            payload={"migration_id": migration_id, "dry_run_passed": dry_run_passed},
        )
        return {"status": "PAUSED_FOR_HUMAN_DECISION", "migration_id": migration_id}

    if tool_context.tool_confirmation.confirmed:
        return {"status": "EXECUTED", "migration_id": migration_id}
    return {"status": "REJECTED_BY_HUMAN", "migration_id": migration_id}

print("Configured MCP Toolset (allowlisted) + HITL Confirmation Gate.")


Configured MCP Toolset (allowlisted) + HITL Confirmation Gate.
